# 0602 Research Investigation Plan
## CNN-Transformer: Baseline vs. Event-Based Pipeline Audit
---


## Part 0 — Confirmed Code Differences (Ground Truth)
Before hypotheses, this is what the code actually shows:

| Dimension | Baseline (`0407`) | Experimental (`copy`) |
|---|---|---|
| **Detector** | `ApexWindowDetector(percentile=95, prominence=0.5, max_window=512)` | `ApexPhaseSpotterROI()` (default params: distance=5, prominence=0.1) |
| **Detector input** | Precomputed magnitudes from `.npz` | Calls `flow_to_magnitude_signal(flow)` on precomputed flow |
| **`phase_mode`** | `"onset_to_apex"` (hardcoded) | `PHASE_MODE = "onset_to_apex"` (via variable, configurable) |
| **Model pooling** | `pooling="mean"` (default) | `pooling="attn"` (explicit) |
| **Augmentation** | `scale_range`, `noise_std` only | Adds `jitter_frames=3`, `dropout_p=0.15` (channel dropout) |
| **Detector prominence** | 0.5 | 0.1 (5× more sensitive) |
| **`MAX_SEQ_LEN`** | `MAX_SEQ_LEN_CAP = 512` (direct) | `MAX_SEQ_LEN_CAP = 512` (same, but computed via percentile path) |
| **Spotter constants** | N/A | `SPOTTER_DISTANCE_THRESHOLD=11`, `SPOTTER_PROMINENCE_THRESHOLD=0.1`, `SPOTTER_DETECT_INTERVAL=3` (declared but unused in `ApexPhaseSpotterROI.__init__`!) |

> **Critical finding 1**: `SPOTTER_DISTANCE_THRESHOLD=11` and `SPOTTER_PROMINENCE_THRESHOLD=0.1` are declared as config constants in the experimental notebook but `ApexPhaseSpotterROI()` is instantiated with **default constructor** (distance=5, prominence=0.1). The distance config is silently ignored. This is a likely misconfiguration bug.

> **Critical finding 2**: `ApexPhaseSpotterROI.detect_windows()` calls `flow_to_magnitude_signal(flow)` to derive a 1D signal from the precomputed `.npz` flow — it does **not** re-run landmark detection or optical flow on raw video. The full per-frame landmark+align+TVL1 pipeline only runs in `process(video_path)`. The two detectors therefore operate on the same magnitude domain but with different peak-detection algorithms and thresholds.


## Part 1 — Architectural Comparison
### 1.1 Data Flow (Both Pipelines Share)
```
.npz file (T, N_roi=5, 2, H=64, W=64)  [float16 on disk → float32 in memory]
        ↓  _load_flow()
AnxietyDatasetBase._build_subject_sample()
  → merges all clips for a subject into (T_total, 5, 2, 64, 64)
  → _detect_windows(flow) → [(l, apex, r), ...]
        ↓
WindowSelector(phase_includes=["onset","apex"])
  → concatenates onset+apex frames across all windows
        ↓
BehavioralFeatures()
  → (T_selected, 5, 2, 64, 64) → (T_selected, 47)
        ↓
PadAndMask(max_len=512)
  → (512, 47) + bool mask
        ↓
AugmentFlow(training=True/False)
        ↓
CNN_Transformer(in_channels=47, d_model=64, nhead=4, num_layers=2)
```

### 1.2 Where They Diverge
**Window Detection Step**:
| | Baseline | Experimental |
|---|---|---|
| Algorithm | `detect_windows_from_signal()` with `percentile=95, prominence=0.5` | `ApexPhaseSpotterROI.detect_windows()` → `flow_to_magnitude_signal()` + `ApexSmoother.smooth()` + `mean+std` threshold + top-10 peaks |
| Height threshold | percentile-based (hard) | `mean + std` (adaptive per-clip) |
| Prominence | 0.5 (high = selective) | 0.1 (low = permissive) |
| Peak selection | percentile filter | top-k=10 |

**Pooling**:
- Baseline: masked global average pooling (stable, uniform weighting)
- Experimental: learned attention pooling (1 linear layer) — adds ~512 parameters but more importantly changes gradient flow

**Augmentation**:
- Experimental adds `jitter_frames=3` (temporal jitter) and `channel_dropout=0.15`


## Part 2 — Data Structure Analysis
### Predicted Structural Differences
Based on the prominence difference (0.5 → 0.1):
- **ApexPhaseSpotterROI** will detect **more peaks per clip** (lower prominence bar)
- More peaks → more windows → more frames selected by `WindowSelector`
- More frames selected → **lower padding ratio** (less padding to 512)
- Lower padding → but also potentially **noisier windows** (false apex detections)
- The `mean+std` threshold is adaptive: clips with low-motion will have a low threshold, potentially selecting noise frames


## Part 3 — Root Cause Analysis for Overfitting
### Ranked Causes
1. **Data**: Permissive detection (prominence=0.1) includes false apex windows, creating noisy training signal (High likelihood)
2. **Model**: Attention pooling overfits to training-specific temporal patterns (High likelihood)
3. **Engineering**: `SPOTTER_DISTANCE_THRESHOLD=11` config is never passed to `ApexPhaseSpotterROI()` (Medium likelihood, confirmed bug)
4. **Data**: Subject-level merging + multiple windows per subject = implicit oversampling of subjects with many detected peaks (Medium-High likelihood)
5. **Temporal**: `flow_to_magnitude_signal()` path may produce a different magnitude signal than what `ApexWindowDetector` uses (Medium likelihood)


## Part 4 — GPU Utilization Analysis
Both notebooks use `num_workers=0`. This is the **primary bottleneck**.
- `np.load()` for ~5 clips × ~43MB = I/O bound
- `BehavioralFeatures._extract()` calls `circvar` which uses scipy, CPU intensive.
GPU utilization ≈ ~1-5%.
**Fix Priority**: `num_workers ≥ 4` with `persistent_workers=True`.


## Part 5 — Event-Based Pipeline Evaluation
**Option B is the most tractable next step**: use `ApexPhaseSpotterROI`'s smoothed magnitude signal as a soft attention prior — inject it as a positional bias into the Transformer's attention scores. This preserves the full sequence (no information loss) and uses event detection as a **guidance signal** rather than a hard filter.


## Part 6 — Stage 1: Data Audit Script
We will now write a diagnostic script to measure the actual dataset statistics for both detectors without running a full training loop.


In [1]:

from typing import Optional, Sequence, Tuple, List
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

class ApexWindowDetector:
    def __init__(
        self,
        percentile: float = 70,
        prominence: float = 0.05,
        min_distance: int = 5,
        ratio: float = 0.30,
        min_window: int = 3,
        max_window: int = 200,
        context: int = 5,
        smooth_sigma: float = 1.5,
    ):
        self.percentile = float(percentile)
        self.prominence = float(prominence)
        self.min_distance = int(min_distance)
        self.ratio = float(ratio)
        self.min_window = int(min_window)
        self.max_window = int(max_window)
        self.context = int(context)
        self.smooth_sigma = float(smooth_sigma)

    def detect_windows(
        self,
        flow: np.ndarray,
        phase_mode: str = "onset_to_apex",
        selected_rois: Optional[Sequence[str]] = None,
    ) -> Tuple[List[Tuple[int, int, int]], dict]:
        T = flow.shape[0]
        dx = flow[:, :, 0, :, :].mean(axis=(2, 3))
        dy = flow[:, :, 1, :, :].mean(axis=(2, 3))

        if selected_rois is not None:
            from src.constants.index import ROI_ORDER_DEFAULT
            roi_idx = np.array([ROI_ORDER_DEFAULT.index(r) for r in selected_rois])
            dx = dx[:, roi_idx]
            dy = dy[:, roi_idx]

        magnitude_1d = np.sqrt(dx**2 + dy**2).mean(axis=1)
        apex_signal = np.log1p(magnitude_1d)

        epsilon = np.percentile(apex_signal, 10)
        apex_signal_clean = apex_signal.copy()
        apex_signal_clean[apex_signal_clean < epsilon] = 0.0

        smoothed = gaussian_filter1d(apex_signal_clean, sigma=self.smooth_sigma)
        threshold = max(np.percentile(smoothed, self.percentile), np.std(smoothed) * 0.5)
        peaks, _ = find_peaks(
            smoothed,
            height=threshold,
            prominence=self.prominence,
            distance=self.min_distance,
        )

        if len(peaks) == 0:
            return [], {"valid": False, "reason": "no_peaks"}

        windows: List[Tuple[int, int, int]] = []
        for p in peaks:
            if phase_mode == "onset_to_apex":
                left = self._find_onset(smoothed, p)
                right = p + 1
            else:
                left, right = self._find_phase(smoothed, p)

            length = right - left
            if length < self.min_window:
                continue
            if length > self.max_window:
                half = self.max_window // 2
                left = max(0, p - half)
                right = min(T, p + half)

            left = max(0, left - self.context)
            if phase_mode != "onset_to_apex":
                right = min(T, right + self.context)

            windows.append((left, int(p), right))

        if not windows:
            return [], {"valid": False, "reason": "no_valid_windows", "num_peaks": len(peaks)}

        apex_vals = smoothed[[p for _, p, _ in windows]]
        confidence = float(np.mean(np.abs(apex_vals)) / (np.mean(np.abs(smoothed)) + 1e-6))
        return windows, {"valid": True, "num_peaks": len(peaks), "num_windows": len(windows), "confidence": confidence}

    def _find_onset(self, signal: np.ndarray, p: int) -> int:
        peak_val = signal[p]
        left = p
        while left > 1:
            if signal[left] < peak_val * self.ratio:
                break
            if signal[left] < signal[left - 1]:
                break
            left -= 1
        return left

    def _find_phase(self, signal: np.ndarray, p: int) -> Tuple[int, int]:
        T = len(signal)
        peak_val = signal[p]
        left = p
        while left > 1:
            if signal[left] < peak_val * self.ratio:
                break
            if signal[left] < signal[left - 1]:
                break
            left -= 1
        right = p
        while right < T - 2:
            if signal[right] < peak_val * self.ratio:
                break
            if signal[right] < signal[right + 1]:
                break
            right += 1
        return left, right

import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
import numpy as np
from src.apex.modules import ApexPhaseSpotterROI
from src.dataset.modules.flow_roi_dataset import FlowROIDataset

# Config
ANNOTATION_PATH = Path("/home/inadio/datasets/anxiety_raw/annotations-v10-clips.csv")
df = pd.read_csv(ANNOTATION_PATH)
if "npy_path" not in df.columns and "cache_path" in df.columns:
    df["npy_path"] = df["cache_path"]
if "is_valid" in df.columns:
    df = df[df["is_valid"]].copy()

# Setup detectors
baseline_detector = ApexWindowDetector(percentile=95, prominence=0.5, max_window=512)
experimental_detector = ApexPhaseSpotterROI() # The currently bugged one
fixed_experimental_detector = ApexPhaseSpotterROI(distance_threshold=11, prominence_threshold=0.1)

print("Setup complete. Ready to audit dataset.")



ImportError: cannot import name 'ApexWindowDetector' from 'src.dataset.modules.window_selector' (/home/inadio/skripkir/pulse-live/combinations-notebooks/../src/dataset/modules/window_selector.py)

### Stage 1: Full Data Audit Script
Run this cell to scan the dataset using all three detectors and compare their statistics. This will confirm or reject H1 (noisy windows) and H3 (config bug).


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

def audit_detector(name, detector, meta_df, max_rows=200):
    print(f"\n--- Auditing {name} ---")
    processed = 0
    ok = 0
    no_peaks = 0
    n_windows = []
    
    label_valid = defaultdict(int)
    label_total = defaultdict(int)
    
    # We will use a dummy FlowROIDataset just for the _load_flow logic
    dummy_ds = FlowROIDataset(metadata_df=meta_df.head(1), transform=None, detector=detector)

    for _, row in meta_df.iterrows():
        if max_rows and processed >= max_rows: break
        
        npy_path = str(row.get("npy_path", "")).strip()
        label = str(row.get("label", "unknown"))
        label_total[label] += 1
        
        try:
            flow = dummy_ds._load_flow(npy_path)
            # detect_windows returns (windows, meta)
            windows, meta = detector.detect_windows(flow, phase_mode="onset_to_apex")
            processed += 1
            
            if not meta.get("valid", False):
                if meta.get("reason") == "no_peaks":
                    no_peaks += 1
                continue
                
            ok += 1
            label_valid[label] += 1
            n_windows.append(len(windows))
            
        except Exception as e:
            # handle missing files or loading errors
            pass
            
    print(f"Processed clips: {processed}")
    print(f"Valid clips (found peaks): {ok} ({ok/max(1,processed)*100:.1f}%)")
    print(f"No peaks found: {no_peaks}")
    print(f"Avg windows per valid clip: {np.mean(n_windows):.2f}" if n_windows else "No windows found")
    
    for lbl in label_total.keys():
        tot = label_total[lbl]
        val = label_valid[lbl]
        print(f"  Valid rate [{lbl}]: {val/max(1,tot)*100:.1f}%")

# Audit Train Split
train_df = df[df['fold'] != 4] if 'fold' in df.columns else df.sample(min(200, len(df)))

audit_detector("Baseline Detector", baseline_detector, train_df)
audit_detector("Experimental (Bugged) Detector", experimental_detector, train_df)
audit_detector("Fixed Experimental Detector", fixed_experimental_detector, train_df)



### Stage 2: Applying the Engineering Fixes
Before running new training sessions, we apply the fixed detector and enable `num_workers=4` with `persistent_workers=True` to solve the GPU utilization bottleneck.


In [ ]:
# Fix 1: Use the corrected detector
detector = ApexPhaseSpotterROI(
    distance_threshold=11, 
    prominence_threshold=0.1
)

# Fix 2: Optimize DataLoader for GPU utilization
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True, # Keeps workers alive between epochs
    collate_fn=collate_fn,
)



### Stage 3 & 4: Ablation Studies Framework
To run the single-variable ablations, modify the variables in this cell before running the training loop.


In [ ]:
# Ablation Config
# Set these according to the experiment (E-mean, E-prom05, etc.)

# H2 Test: Mean Pooling vs Attention Pooling
POOLING_MODE = "mean" # Change to "attn" for experimental

# H1 Test: Prominence threshold
PROMINENCE = 0.5 # Change to 0.1 for experimental

# H6 Test: Channel Dropout
CHANNEL_DROPOUT = 0.0 # Change to 0.15 for experimental

# Jitter
JITTER_FRAMES = 0 # Change to 3 for experimental

print(f"Running Ablation:")
print(f"- Pooling: {POOLING_MODE}")
print(f"- Prominence: {PROMINENCE}")
print(f"- Channel Dropout: {CHANNEL_DROPOUT}")
print(f"- Jitter: {JITTER_FRAMES}")



### Stage 5: Event-Based Architecture (Option B - Soft Attention Prior)
If H1 and H2 confirm that event detection creates noisy windows and attention pooling overfits, Option B uses the magnitude signal as a soft attention bias.
This allows the Transformer to look at the whole sequence but pay more attention to the apex peaks without hard-cropping them.


In [ ]:
import torch.nn as nn
from src.models.modules.cnn_transformer.cnn_1d_extractor import CNN1DExtractor
from src.models.modules.cnn_transformer.positional_encoding import PositionalEncoding

class EventGuided_CNN_Transformer(nn.Module):
    def __init__(self, in_channels=47, d_model=64, nhead=4, num_layers=2, num_classes=2, dropout_p=0.5):
        super().__init__()
        self.cnn = CNN1DExtractor(in_channels, out_channels=d_model, pool_size=2, dropout_p=dropout_p)
        self.cnn_refine = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout_p, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x, mask=None, event_scores=None):
        # x: (Batch, Channels, Time)
        # event_scores: (Batch, Time) -> Smooth magnitude from ApexPhaseSpotter
        
        x = self.cnn(x)
        x = x + self.cnn_refine(x)
        
        # Downsample mask and event_scores to match CNN output size
        # (Downsampling logic omitted for brevity)
        
        x = x.permute(0, 2, 1) # (Batch, Time, Channels)
        x = self.pos_encoder(x)
        
        # Inject Event Bias
        # We can add event_scores as a positional bias or use it to scale features
        if event_scores is not None:
            # Scale features by their event probability
            event_bias = event_scores.unsqueeze(-1)
            x = x * (1.0 + event_bias)
            
        x = self.transformer_encoder(x, src_key_padding_mask=mask)
        
        # Masked Mean Pooling
        if mask is not None:
            valid_mask = ~mask
            valid_mask_expanded = valid_mask.unsqueeze(-1).expand_as(x).float()
            x_masked = x * valid_mask_expanded
            denom = valid_mask_expanded.sum(dim=1).clamp(min=1.0)
            x_pooled = x_masked.sum(dim=1) / denom
        else:
            x_pooled = x.mean(dim=1)
            
        return self.classifier(x_pooled)



## Part 7 — Validation Checklist
### Pre-Training Checklist
- [ ] Verify `detector.apex_phase.distance == SPOTTER_DISTANCE_THRESHOLD`
- [ ] Print `avg_windows_per_clip` for train/val split with both detectors
- [ ] Measure `padding_ratio` — if >0.7, sequences are mostly padding (problem)
- [ ] Confirm `valid_rate` is not significantly different between classes

### During Training Checklist
- [ ] Log `train_loss` vs `val_loss` gap per epoch (overfitting onset epoch)
- [ ] Log attention weight entropy (if using attn pooling) — low entropy = peaked = overfitting
- [ ] Monitor GPU utilization — should improve with `num_workers>0`

### Post-Training Checklist
- [ ] T-SNE: do train/val features separate in embedding space?
- [ ] Confusion matrix: is one class consistently misclassified?
- [ ] Threshold search: does optimal threshold shift significantly from 0.5?
- [ ] Sanity check: run inference on training data — if <0.95 acc, model underfits


## Part 8 — Research Roadmap
### Phase 1 — Data Audit (Days 1–2)
Run the Stage 1 diagnostic cell to collect ground-truth statistics. Identify the config bug and confirm its effect.

### Phase 2 — Engineering Fixes (Day 3)
Enable `num_workers=4` + `persistent_workers=True` and fix `SPOTTER_DISTANCE_THRESHOLD` passthrough.

### Phase 3 — Overfitting Diagnosis (Days 4–7)
Run Stage 3 & 4 ablations. Record best val F1 and train/val gap at epoch 50.

### Phase 4 — Event-Based Redesign (Days 8–14)
Based on findings, implement Option B (EventGuided_CNN_Transformer).

### Phase 5 — Final Model Development (Days 15–21)
Full 100-epoch training with best configuration, sweep on event detection parameters, and multi-seed evaluation. Evaluate against test annotations.
